In [13]:
import os
import sys
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from fredapi import Fred

In [2]:
os.environ['JAVA_HOME'] = '/usr/local/sdkman/candidates/java/17.0.10-tem'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [3]:
spark = SparkSession.builder \
    .appName("HMDA_Processing") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 07:20:28 WARN Utils: Your hostname, codespaces-3862d1, resolves to a loopback address: 127.0.0.1; using 10.0.14.72 instead (on interface eth0)
26/03/18 07:20:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 07:20:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
file_path = "2024_lar.txt"

df = spark.read.options(header='True', inferSchema='True', delimiter='|') \
    .csv(file_path)

In [5]:
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")

root
 |-- activity_year: integer (nullable = true)
 |-- lei: string (nullable = true)
 |-- derived_msa_md: integer (nullable = true)
 |-- state_code: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- census_tract: string (nullable = true)
 |-- conforming_loan_limit: string (nullable = true)
 |-- derived_loan_product_type: string (nullable = true)
 |-- derived_dwelling_category: string (nullable = true)
 |-- derived_ethnicity: string (nullable = true)
 |-- derived_race: string (nullable = true)
 |-- derived_sex: string (nullable = true)
 |-- action_taken: integer (nullable = true)
 |-- purchaser_type: integer (nullable = true)
 |-- preapproval: integer (nullable = true)
 |-- loan_type: integer (nullable = true)
 |-- loan_purpose: integer (nullable = true)
 |-- lien_status: integer (nullable = true)
 |-- reverse_mortgage: integer (nullable = true)
 |-- open_end_line_of_credit: integer (nullable = true)
 |-- business_or_commercial_purpose: integer (nullable = true)


[Stage 2:=======================================================> (34 + 1) / 35]

Tổng số bản ghi: 12259095


In [6]:
df.show(5)

26/03/18 07:22:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+--------------------+--------------+----------+-----------+------------+---------------------+-------------------------+-------------------------+--------------------+--------------------+-----------------+------------+--------------+-----------+---------+------------+-----------+----------------+-----------------------+------------------------------+-----------+----------------------------+-------------+-----------+------------+----------------+---------------------+-------------------+---------------+--------------+---------+-----------------------+-----------------+---------------------+---------------------+---------------+----------------------------+--------------+-------------------+--------------+---------------------------------------+----------------------------------------+-----------+----------------------------+------+--------------------+---------------------------+------------------------------+---------------------+---------------------+------------------

In [7]:
selected_columns = [
    "activity_year", "lei", "state_code", "loan_type", "loan_purpose", 
    "loan_amount", "interest_rate", "property_value", "income", 
    "debt_to_income_ratio", "combined_loan_to_value_ratio", "loan_term", 
    "intro_rate_period", "interest_only_payment", "balloon_payment", 
    "negative_amortization", "action_taken", "denial_reason_1", 
    "derived_race", "derived_sex"
]

In [15]:
df_cleaned = df.select(*selected_columns) \
    .withColumn("loan_amount", F.expr("try_cast(loan_amount AS double)")) \
    .withColumn("interest_rate", F.expr("try_cast(interest_rate AS double)")) \
    .withColumn("property_value", F.expr("try_cast(property_value AS double)")) \
    .withColumn("income", F.expr("try_cast(income AS double)")) \
    .filter(F.col("action_taken") == 1)

In [16]:
df_cleaned.createOrReplaceTempView("v_credit_risk_raw")

In [17]:
query1 = spark.sql("""
    SELECT 
        state_code, 
        COUNT(*) as total_loans,
        ROUND(AVG(loan_amount), 2) as avg_loan_size,
        ROUND(AVG(interest_rate), 2) as avg_interest_rate
    FROM v_credit_risk_raw
    GROUP BY state_code
    ORDER BY total_loans DESC
    LIMIT 10
""")
query1.show()

[Stage 7:=======================================================> (34 + 1) / 35]

+----------+-----------+-------------+-----------------+
|state_code|total_loans|avg_loan_size|avg_interest_rate|
+----------+-----------+-------------+-----------------+
|        CA|     511273|    566399.25|             7.32|
|        TX|     494658|    350446.25|             6.74|
|        FL|     476449|    359272.44|             7.05|
|        NC|     256799|    299331.87|             7.01|
|        OH|     246244|     218364.1|             7.46|
|        PA|     246037|    226384.34|             7.12|
|        GA|     227742|    312669.86|             7.09|
|        NY|     208819|    445658.27|             7.11|
|        IL|     207606|    273376.49|             7.17|
|        MI|     206122|    211530.36|             7.42|
+----------+-----------+-------------+-----------------+



In [19]:
query2 = spark.sql("""
    SELECT 
        lei, 
        loan_amount, 
        income, 
        debt_to_income_ratio, 
        interest_rate
    FROM v_credit_risk_raw
    WHERE interest_rate > 6.0 
      AND debt_to_income_ratio IN ('45', '46', '47', '48', '49', '50%-60%', '>60%')
    ORDER BY loan_amount DESC
""")
query2.show()

[Stage 11:======================================================> (34 + 1) / 35]

+--------------------+-----------+-------+--------------------+-------------+
|                 lei|loan_amount| income|debt_to_income_ratio|interest_rate|
+--------------------+-----------+-------+--------------------+-------------+
|549300GWD9H4FQ2VR805|   2.3105E7| 6286.0|                  47|         6.25|
|7H6GLXDRUGQFU57RNE97|   2.2405E7|    0.0|                >60%|        8.125|
|549300GWD9H4FQ2VR805|   2.2005E7| 9249.0|                  45|          6.5|
|7H6GLXDRUGQFU57RNE97|   1.8555E7|    0.0|                  49|        6.125|
|5493008PQSMR41Y70C36|   1.6905E7| 6845.0|             50%-60%|         9.31|
|5493008PQSMR41Y70C36|   1.6505E7| 5124.0|             50%-60%|          6.5|
|OYWNLMHNBQQ7BAH3EE86|   1.6005E7|  567.0|                >60%|         6.73|
|7H6GLXDRUGQFU57RNE97|   1.4955E7|24864.0|                  48|         6.25|
|2549009X2AG1P20YAJ63|   1.4705E7|   34.0|                >60%|          9.5|
|2549009X2AG1P20YAJ63|   1.3775E7| 6396.0|                  46| 

In [21]:
spark.sql("""
    SELECT count(*)
    FROM v_credit_risk_raw
""").show()

[Stage 13:======================================================> (34 + 1) / 35]

+--------+
|count(1)|
+--------+
| 6197069|
+--------+



In [22]:
df_cleaned.repartition(4)

DataFrame[activity_year: int, lei: string, state_code: string, loan_type: int, loan_purpose: int, loan_amount: double, interest_rate: double, property_value: double, income: double, debt_to_income_ratio: string, combined_loan_to_value_ratio: string, loan_term: string, intro_rate_period: string, interest_only_payment: int, balloon_payment: int, negative_amortization: int, action_taken: int, denial_reason_1: int, derived_race: string, derived_sex: string]

In [23]:
output_path = "processed_hmda_2024"

df_cleaned.write \
    .mode("overwrite") \
    .partitionBy("state_code") \
    .parquet(output_path)

print(f"✅ Đã lưu dữ liệu tại: {output_path}")

✅ Đã lưu dữ liệu tại: processed_hmda_2024


In [26]:
with open('fred-api-key.txt', 'r') as f:
    FRED_API_KEY = f.read().strip()

fred = Fred(api_key=FRED_API_KEY)

In [29]:
s = fred.get_series('FEDFUNDS')
df_pd = pd.DataFrame(s, columns=['fed_rate']).reset_index()
df_pd.rename(columns={'index': 'observation_date'}, inplace=True)

In [30]:
df_fred = spark.createDataFrame(df_pd)

In [31]:
df_fred = df_fred.withColumn("activity_year", F.year("observation_date"))

df_fred.show(5)

[Stage 17:>                                                         (0 + 1) / 1]

+-------------------+--------+-------------+
|   observation_date|fed_rate|activity_year|
+-------------------+--------+-------------+
|1954-07-01 00:00:00|     0.8|         1954|
|1954-08-01 00:00:00|    1.22|         1954|
|1954-09-01 00:00:00|    1.07|         1954|
|1954-10-01 00:00:00|    0.85|         1954|
|1954-11-01 00:00:00|    0.83|         1954|
+-------------------+--------+-------------+
only showing top 5 rows


In [33]:
df_fred.orderBy("activity_year", "observation_date", ascending=False).show()

+-------------------+--------+-------------+
|   observation_date|fed_rate|activity_year|
+-------------------+--------+-------------+
|2026-02-01 00:00:00|    3.64|         2026|
|2026-01-01 00:00:00|    3.64|         2026|
|2025-12-01 00:00:00|    3.72|         2025|
|2025-11-01 00:00:00|    3.88|         2025|
|2025-10-01 00:00:00|    4.09|         2025|
|2025-09-01 00:00:00|    4.22|         2025|
|2025-08-01 00:00:00|    4.33|         2025|
|2025-07-01 00:00:00|    4.33|         2025|
|2025-06-01 00:00:00|    4.33|         2025|
|2025-05-01 00:00:00|    4.33|         2025|
|2025-04-01 00:00:00|    4.33|         2025|
|2025-03-01 00:00:00|    4.33|         2025|
|2025-02-01 00:00:00|    4.33|         2025|
|2025-01-01 00:00:00|    4.33|         2025|
|2024-12-01 00:00:00|    4.48|         2024|
|2024-11-01 00:00:00|    4.64|         2024|
|2024-10-01 00:00:00|    4.83|         2024|
|2024-09-01 00:00:00|    5.13|         2024|
|2024-08-01 00:00:00|    5.33|         2024|
|2024-07-0

In [36]:
df_fred_yearly = df_fred.groupBy("activity_year") \
    .agg(F.round(F.avg("fed_rate"), 2).alias("avg_fed_rate"))

df_fred_yearly.orderBy("activity_year", ascending=False).show()

+-------------+------------+
|activity_year|avg_fed_rate|
+-------------+------------+
|         2026|        3.64|
|         2025|        4.21|
|         2024|        5.14|
|         2023|        5.02|
|         2022|        1.68|
|         2021|        0.08|
|         2020|        0.38|
|         2019|        2.16|
|         2018|        1.83|
|         2017|         1.0|
|         2016|         0.4|
|         2015|        0.13|
|         2014|        0.09|
|         2013|        0.11|
|         2012|        0.14|
|         2011|         0.1|
|         2010|        0.17|
|         2009|        0.16|
|         2008|        1.93|
|         2007|        5.02|
+-------------+------------+
only showing top 20 rows


In [37]:
df_silver = df_cleaned.join(df_fred_yearly, on="activity_year", how="left")

null_count = df_silver.filter(F.col("avg_fed_rate").isNull()).count()
print(f"⚠️ Số lượng dòng bị thiếu dữ liệu FRED: {null_count}")

if null_count > 0:
    df_silver.filter(F.col("avg_fed_rate").isNull()).select("activity_year", "lei").show(5)

[Stage 31:>                                                         (0 + 1) / 1]

⚠️ Số lượng dòng bị thiếu dữ liệu FRED: 0


In [38]:
# Loại bỏ các dòng NULL để chuẩn bị tính toán "Spread"
df_silver = df_silver.dropna(subset=["avg_fed_rate"])

# Bây giờ tính toán chỉ số rủi ro: Spread = Lãi suất vay - Lãi suất FED
df_silver = df_silver.withColumn("interest_rate_spread", 
                                 F.round(F.col("interest_rate") - F.col("avg_fed_rate"), 2))

# Xem kết quả cuối cùng
df_silver.select("activity_year", "lei", "interest_rate", "avg_fed_rate", "interest_rate_spread").show(10)

+-------------+--------------------+-------------+------------+--------------------+
|activity_year|                 lei|interest_rate|avg_fed_rate|interest_rate_spread|
+-------------+--------------------+-------------+------------+--------------------+
|         2024|Q7C315HKI8VX0SSKBS64|         8.75|        5.14|                3.61|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|          8.5|        5.14|                3.36|
|         2024|Q7C315HKI8VX0SSKBS64|         8.75|        5.14|  